In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/image/images.zip -d /content/images.zip
!unzip -q images.zip

In [ ]:
!rm /content/images.zip

In [ ]:
import os
A=os.listdir('/content')
A

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
s='/content/images/*'
path=glob.glob(s)
import re
def sorted_alphanumeric(data):
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [ convert(c) for c in re.split('([0-9]+)', key) ]
    return sorted(data, key=alphanum_key)

path1=sorted_alphanumeric(path)
path1 = [item.replace('p','_') for item in path1]
data=pd.read_excel('/content/drive/MyDrive/image/data_c.xlsx')
data
len(path1)
print(path1)
type(path1)
path1=np.array(path1)
path1.shape
path1[0]

In [ ]:
path1=np.array(path1)
data=np.array(data)
data
len(data)

In [ ]:
df=pd.DataFrame({'imgpath':path1,'Porosity':data[:,0],'throat radius':data[:,1],'pore radius':data[:,2],'pore_connection_number':data[:,3],'pore shape factor':data[:,4]})

In [ ]:
df

In [ ]:
df_new=df[df['Porosity'] <0.35 ]
df_new.shape
df_new

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data2=np.array(df_new)
scaler = StandardScaler()
data1= scaler.fit_transform(data2[:,1:])
print(data1)
print(type(data1))
data1.shape

In [ ]:
data2

In [ ]:
df=pd.DataFrame({'imgpath':df_new['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

In [ ]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.15,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

In [ ]:
def Data_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img = []
                y_batch=[]
                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    Porosity=np.array(df.Porosity)[idd]
                    throat_radius=np.array(df['throat radius'])[idd]
                    pore_radius=np.array(df['pore radius'])[idd]
                    pore_connection_number=np.array(df.pore_connection_number)[idd]
                    pore_shape_factor=np.array(df['pore shape factor'])[idd]
                    y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
                    y=np.array([y_1])
                    x_batch_img.append(img_1)
                    y_batch.append(y)


                x_batch_img = np.array(x_batch_img)
                y_batch= np.array(y_batch)
                y_batch=y_batch.reshape(-1,5)
              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img , y_batch

In [ ]:
def Data_predict_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img_2 = []

                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    x_batch_img_2.append(img_1)



                x_batch_img_2 = np.array(x_batch_img_2)

              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img_2

In [ ]:
batch_size=10
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
train_gen_pre=Data_predict_generator(df_train,batch_size)
valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import math
ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
nbatches_train=math.ceil(df_train.shape[0]/batch_size)
nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
print(nbatches_valid,nbatches_train,nbatches_test)

In [ ]:
from keras.models import Sequential
from keras.layers import Conv3D,MaxPooling3D,Dropout,Flatten,Dense,BatchNormalization,ReLU
from tensorflow.keras import initializers
from keras.callbacks import *
import tensorflow as tf

In [ ]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(11,11,11),input_shape=(100,100,100,1),padding="same",activation='relu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(7,7,7),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(5,5,5),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

In [ ]:
import os
checkpoint_path="/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/11*7*5*3*3/training_model_weights/cp-{epoch:03d}.ckpt"
#os.makedirs("/content/drive/MyDrive/image/porosity_filtering/training_5layers", exist_ok=True)
#ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/image/porosity_filtering/training_model_5layers/weights.{epoch:02d}-{val_loss:.2f}.hdf5', monitor='val_loss')
cp_callback=ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1)

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/11*7*5*3*3/training_192.log')

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger])

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/training_model_weights/cp-064.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=64)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/training_model_weights/cp-084.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=84)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/11*7*5*3*3/training_model_weights/cp-100.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=100)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/training_model_weights/cp-142.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=142)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/training_model_weights/cp-191.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=191)

In [ ]:
#**********************************************predicting***********************************************

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/kernel_dimension/11*7*5*3*3/training_model_weights/cp-199.ckpt'
model.load_weights(wieght)

In [ ]:
model.evaluate_generator(test_gen,  nbatches_test, workers=1)

In [ ]:
y=model.predict_generator(
    test_gen_pre,
    verbose=1,
    steps=nbatches_test,
    callbacks=None,
    max_queue_size=10,
    workers=1,
    use_multiprocessing=False)

In [ ]:
y_1=pd.DataFrame({'porosity':((y[:,0]*df_new['Porosity'].std())+df_new['Porosity'].mean()),
                  'throat radius':((y[:,1]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((y[:,2]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((y[:,3]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((y[:,4]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
df_test_1=np.array(df_test)
df_test_2=pd.DataFrame({'porosity':((df_test_1[:,1]*df_new.Porosity.std())+df_new.Porosity.mean()),
                  'throat radius':((df_test_1[:,2]*df_new['throat radius'].std())+df_new['throat radius'].mean()),
                'pore radius':((df_test_1[:,3]*df_new['pore radius'].std())+df_new['pore radius'].mean()),
                'pore_connection_number':((df_test_1[:,4]*df_new.pore_connection_number.std())+df_new.pore_connection_number.mean()),
                 'pore shape factor':((df_test_1[:,5]*df_new['pore shape factor'].std())+df_new['pore shape factor'].mean())})

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['porosity'])
y_values = np.array(y_1['porosity'])
r_squared_porosity=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_porosity)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore shape factor'])
y_values = np.array(y_1['pore shape factor'])
r_squared_pore_shape_factor=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_shape_factor)

In [ ]:
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['throat radius'])
y_values = np.array(y_1['throat radius'])
r_squared_throat_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_throat_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore radius'])
y_values = np.array(y_1['pore radius'])
r_squared_pore_radius=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_radius)

In [ ]:
#R^2_porosity
from sklearn.metrics import r2_score
x_values = np.array(df_test_2['pore_connection_number'])
y_values = np.array(y_1['pore_connection_number'])
r_squared_pore_connection_number=r2_score(x_values, y_values)
#orrelation_matrix = np.corrcoef(x_values, y_values)
#correlation_xy = correlation_matrix[0,1]
#r_squared_porosity = correlation_xy**2
print(r_squared_pore_connection_number)